# Part C — Complete Mage Iris Pipeline (Colab)



In [3]:

import subprocess

# Install Python 3.12 virtual environment support
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "python3.12-venv"],
    check=True
)

# Create a fresh environment
subprocess.run(
    ["python3.12", "-m", "venv", "/content/mage_env"],
    check=True
)

PYTHON = "/content/mage_env/bin/python"
MAGE = "/content/mage_env/bin/mage"

# Install Mage and required libraries
subprocess.run(
    [PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True
)

subprocess.run(
    [PYTHON, "-m", "pip", "install",
     "mage-ai==0.9.80",
     "scikit-learn", "pandas", "numpy",
     "matplotlib", "seaborn"],
    check=True
)

print("Mage installation completed.")


Mage installation completed.


In [4]:

import subprocess
from pathlib import Path

PROJECT = Path("/content/iris_project")
MAGE = "/content/mage_env/bin/mage"

subprocess.run(
    [MAGE, "init", str(PROJECT), "--project-type", "standalone"],
    check=True
)

print("Mage project initialized:", PROJECT.exists())
print("Project folders:")
for item in PROJECT.iterdir():
    print("-", item.name)


Mage project initialized: True
Project folders:
- charts
- .gitignore
- pipelines
- data_loaders
- io_config.yaml
- __init__.py
- requirements.txt
- metadata.yaml
- extensions
- __pycache__
- interactions
- dbt
- custom
- scratchpads
- data_exporters
- utils
- transformers


In [5]:
from pathlib import Path
import subprocess, os, json, time, textwrap, yaml
import pandas as pd

PROJECT = Path("/content/iris_project")
PYTHON = "/content/mage_env/bin/python"
MAGE = "/content/mage_env/bin/mage"

assert Path(PYTHON).exists(), "Python 3.12 Mage environment is missing."
assert Path(MAGE).exists(), "Mage CLI is missing."
assert PROJECT.exists(), "Initialize /content/iris_project first."
os.chdir("/content")
print(subprocess.check_output([PYTHON, "--version"], text=True).strip())
print("Mage project:", PROJECT)

Python 3.12.3
Mage project: /content/iris_project


## 1. Create Iris block file

In [6]:
for folder in ["data_loaders", "transformers", "data_exporters", "pipelines/iris_classifier"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

files = {
"data_loaders/load_iris.py": r"""
from sklearn.datasets import load_iris
if 'data_loader' not in globals():
    from mage_ai.data_preparation.decorators import data_loader
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@data_loader
def load_data(*args, **kwargs):
    iris = load_iris(as_frame=True)
    df = iris.frame
    print("Rows loaded:", len(df))
    return df

@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert output.shape[1] == 5
""",
"transformers/prepare_data.py": r"""
import pandas as pd
from sklearn.model_selection import train_test_split
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def transform(df, *args, **kwargs):
    df = df.dropna().copy()
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df['target']
    )
    train_df['split'] = 'train'
    test_df['split'] = 'test'
    result = pd.concat([train_df, test_df]).sort_index()
    print("Train rows:", len(train_df), "| Test rows:", len(test_df))
    return result

@test
def test_output(output, *args):
    assert 'split' in output.columns
    assert (output['split'] == 'train').sum() == 120
    assert (output['split'] == 'test').sum() == 30
""",
"transformers/train_model.py": r"""
import joblib
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

FEATURES = ['sepal length (cm)', 'sepal width (cm)',
            'petal length (cm)', 'petal width (cm)']

@transformer
def train(df, *args, **kwargs):
    max_depth = int(kwargs.get('max_depth', 3))
    train_df = df[df['split'] == 'train']
    test_df = df[df['split'] == 'test'].copy()
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(train_df[FEATURES], train_df['target'])
    test_df['predicted'] = model.predict(test_df[FEATURES])
    accuracy = (test_df['predicted'] == test_df['target']).mean()
    test_df['test_accuracy'] = accuracy
    test_df['max_depth'] = max_depth
    print(f"max_depth={max_depth} | Test accuracy={accuracy:.4f}")
    joblib.dump(model, '/content/iris_project/iris_model.pkl')
    return test_df

@test
def test_accuracy(output, *args):
    accuracy = (output['predicted'] == output['target']).mean()
    assert accuracy >= 0.8, f"Model accuracy {accuracy:.3f} is below 80%"
""",
"data_exporters/save_results.py": r"""
from pathlib import Path
if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@data_exporter
def export_data(df, *args, **kwargs):
    path = Path('/content/iris_project/iris_predictions.csv')
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} predictions to {path}")

@test
def test_export(output, *args):
    path = Path('/content/iris_project/iris_predictions.csv')
    assert path.exists()
    assert path.stat().st_size > 0
""",
"transformers/evaluate_model.py": r"""
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def evaluate(df, *args, **kwargs):
    labels = sorted(df['target'].unique())
    cm = confusion_matrix(df['target'], df['predicted'], labels=labels)
    print("Confusion matrix (rows=true, columns=predicted):")
    print(pd.DataFrame(cm, index=labels, columns=labels))
    metrics = pd.DataFrame([{
        'accuracy': accuracy_score(df['target'], df['predicted']),
        'precision_macro': precision_score(df['target'], df['predicted'], average='macro', zero_division=0),
        'recall_macro': recall_score(df['target'], df['predicted'], average='macro', zero_division=0),
        'f1_macro': f1_score(df['target'], df['predicted'], average='macro', zero_division=0),
    }])
    print("Metrics:")
    print(metrics.to_string(index=False))
    return df

@test
def test_evaluation(output, *args):
    assert 'predicted' in output.columns
    assert len(output) == 30
""",
"transformers/validate_iris.py": r"""
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def validate_iris(df, *args, **kwargs):
    assert not df.isna().any().any(), "Iris data contains missing values"
    assert df['target'].nunique() == 3, "Expected exactly 3 classes"
    print("Validation passed: no missing values and exactly 3 classes.")
    return df

@test
def test_iris_validation(output, *args):
    assert output.isna().sum().sum() == 0
    assert output['target'].nunique() == 3
"""
}
for rel, src in files.items():
    p = PROJECT / rel
    p.write_text(textwrap.dedent(src).strip() + "\n")
    print("Created", p)

Created /content/iris_project/data_loaders/load_iris.py
Created /content/iris_project/transformers/prepare_data.py
Created /content/iris_project/transformers/train_model.py
Created /content/iris_project/data_exporters/save_results.py
Created /content/iris_project/transformers/evaluate_model.py
Created /content/iris_project/transformers/validate_iris.py


## 2. Write initial C1 metadata (four-block chain)

In [7]:
pipeline_dir = PROJECT / "pipelines" / "iris_classifier"
meta = {
 "blocks": [
  {"name":"load_iris","type":"data_loader","uuid":"load_iris","upstream_blocks":[],"downstream_blocks":["prepare_data"],"all_upstream_blocks_executed":True,"status":"not_executed"},
  {"name":"prepare_data","type":"transformer","uuid":"prepare_data","upstream_blocks":["load_iris"],"downstream_blocks":["train_model"],"all_upstream_blocks_executed":True,"status":"not_executed"},
  {"name":"train_model","type":"transformer","uuid":"train_model","upstream_blocks":["prepare_data"],"downstream_blocks":["save_results"],"all_upstream_blocks_executed":True,"status":"not_executed"},
  {"name":"save_results","type":"data_exporter","uuid":"save_results","upstream_blocks":["train_model"],"downstream_blocks":[],"all_upstream_blocks_executed":True,"status":"not_executed"}
 ],
 "name":"iris_classifier","type":"python","uuid":"iris_classifier","widgets":[]
}
(pipeline_dir / "metadata.yaml").write_text(yaml.safe_dump(meta, sort_keys=False))
print((pipeline_dir / "metadata.yaml").read_text())

blocks:
- name: load_iris
  type: data_loader
  uuid: load_iris
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  all_upstream_blocks_executed: true
  status: not_executed
- name: prepare_data
  type: transformer
  uuid: prepare_data
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: train_model
  type: transformer
  uuid: train_model
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  all_upstream_blocks_executed: true
  status: not_executed
- name: save_results
  type: data_exporter
  uuid: save_results
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
name: iris_classifier
type: python
uuid: iris_classifier
widgets: []



## 3. Check Mage run syntax

In [8]:
help_result = subprocess.run([MAGE, "run", "--help"], capture_output=True, text=True)
print(help_result.stdout)
if help_result.stderr:
    print(help_result.stderr)

                                                                                
 Usage: mage run [OPTIONS] PROJECT_PATH PIPELINE_UUID                           
                                                                                
 Run pipeline.                                                                  
                                                                                
╭─ Arguments ──────────────────────────────────────────────────────────────────╮
│ *    project_path       TEXT  path of the Mage project that contains the     │
│                               pipeline.                                      │
│                               [default: None]                                │
│                               [required]                                     │
│ *    pipeline_uuid      TEXT  uuid of the pipeline to be run.                │
│                               [default: None]                                │
│                           

## 4. C1 — Run pipeline with block tests

In [9]:
cmd = [MAGE, "run", "iris_project", "iris_classifier", "--test"]
result = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)
print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
assert result.returncode == 0, "C1 pipeline failed; inspect output."

Return code: 1
STDOUT:
 DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 | Test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Saved 30 predictions to /content/iris_project/iris_predictions.csv

STDERR:
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/exec │
│ utors/pipeline_executor.py:82 in execute                                     │
│                                                                              │
│    79 │   │   │   │   )                                                      │
│    80 │   │   # Execute the

AssertionError: C1 pipeline failed; inspect output.

## 5. C2 — Experiment with max_depth 1 to 5
Before every run this cell deletes `iris_predictions.csv`, then records actual accuracy, test outcome and whether the exporter created the file.

In [ ]:
experiment_rows = []
for depth in [1, 2, 3, 4, 5]:
    pred_file = PROJECT / "iris_predictions.csv"
    if pred_file.exists():
        pred_file.unlink()
    cmd = [MAGE, "run", "iris_project", "iris_classifier",
           "--test", "--runtime-vars", json.dumps({"max_depth": depth})]
    r = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)
    print(f"\n===== max_depth={depth} | return={r.returncode} =====")
    print(r.stdout)
    if r.stderr:
        print("STDERR:", r.stderr)
    accuracy = None
    created = pred_file.exists()
    if created:
        out_df = pd.read_csv(pred_file)
        if "test_accuracy" in out_df.columns and len(out_df):
            accuracy = float(out_df["test_accuracy"].iloc[0])
    experiment_rows.append({
        "max_depth": depth,
        "test_accuracy": accuracy,
        "test_accuracy_passed": r.returncode == 0,
        "iris_predictions.csv_created": created,
        "mage_return_code": r.returncode
    })

experiment_table = pd.DataFrame(experiment_rows)
display(experiment_table)

**C2 explanation:** Mage runs `@test` functions after their associated blocks. If `test_accuracy` fails, the train block/run fails and its downstream exporter is not executed. This quality gate prevents invalid predictions from being saved as a successful output. The table above must contain your actual run results.

## 6. C3 — Add `evaluate_model` as a second downstream branch

In [10]:
# Keep save_results as one child and add evaluate_model as another child of train_model.
meta["blocks"][2]["downstream_blocks"] = ["save_results", "evaluate_model"]
meta["blocks"].append({
    "name":"evaluate_model","type":"transformer","uuid":"evaluate_model",
    "upstream_blocks":["train_model"],"downstream_blocks":[],
    "all_upstream_blocks_executed":True,"status":"not_executed"
})
(pipeline_dir / "metadata.yaml").write_text(yaml.safe_dump(meta, sort_keys=False))
print((pipeline_dir / "metadata.yaml").read_text())

blocks:
- name: load_iris
  type: data_loader
  uuid: load_iris
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  all_upstream_blocks_executed: true
  status: not_executed
- name: prepare_data
  type: transformer
  uuid: prepare_data
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: train_model
  type: transformer
  uuid: train_model
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: save_results
  type: data_exporter
  uuid: save_results
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
- name: evaluate_model
  type: transformer
  uuid: evaluate_model
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
name: iris_classifier
type: python
uuid: ir

In [11]:
pred_file = PROJECT / "iris_predictions.csv"
if pred_file.exists():
    pred_file.unlink()
cmd = [MAGE, "run", "iris_project", "iris_classifier",
       "--test", "--runtime-vars", json.dumps({"max_depth": 3})]
r = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)
print("Return code:", r.returncode)
print("STDOUT:\n", r.stdout)
print("STDERR:\n", r.stderr)
assert r.returncode == 0, "Branched pipeline failed."

Return code: 1
STDOUT:
 DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 | Test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows=true, columns=predicted):
    0  1   2
0  10  0   0
1   0  9   1
2   0  0  10
Metrics:
 accuracy  precision_macro  recall_macro  f1_macro
 0.966667         0.969697      0.966667  0.966583
Saved 30 predictions to /content/iris_project/iris_predictions.csv

STDERR:
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/exec │
│ utors/pipeline_executor.py:82 in execute                  

AssertionError: Branched pipeline failed.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

pred_df = pd.read_csv(PROJECT / "iris_predictions.csv")
labels = sorted(pred_df["target"].unique())
cm = confusion_matrix(pred_df["target"], pred_df["predicted"], labels=labels)
confusion_table = pd.DataFrame(cm, index=[f"true_{x}" for x in labels],
                               columns=[f"pred_{x}" for x in labels])
metrics_table = pd.DataFrame([{
    "accuracy": accuracy_score(pred_df["target"], pred_df["predicted"]),
    "precision_macro": precision_score(pred_df["target"], pred_df["predicted"], average="macro", zero_division=0),
    "recall_macro": recall_score(pred_df["target"], pred_df["predicted"], average="macro", zero_division=0),
    "f1_macro": f1_score(pred_df["target"], pred_df["predicted"], average="macro", zero_division=0)
}])
print("Confusion matrix:")
display(confusion_table)
print("Metrics:")
display(metrics_table)

## 7. C4 — Add your own tested block (`validate_iris`)
This transformer is inserted after `load_iris`. It checks that the source data has no missing values and contains exactly three classes.

In [ ]:
# Insert validate_iris between load_iris and prepare_data.
meta["blocks"][0]["downstream_blocks"] = ["validate_iris"]
meta["blocks"][1]["upstream_blocks"] = ["validate_iris"]
meta["blocks"].append({
    "name":"validate_iris","type":"transformer","uuid":"validate_iris",
    "upstream_blocks":["load_iris"],"downstream_blocks":["prepare_data"],
    "all_upstream_blocks_executed":True,"status":"not_executed"
})
(pipeline_dir / "metadata.yaml").write_text(yaml.safe_dump(meta, sort_keys=False))
print((pipeline_dir / "metadata.yaml").read_text())

In [ ]:
pred_file = PROJECT / "iris_predictions.csv"
if pred_file.exists():
    pred_file.unlink()
cmd = [MAGE, "run", "iris_project", "iris_classifier",
       "--test", "--runtime-vars", json.dumps({"max_depth": 3})]
r = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)
print("Return code:", r.returncode)
print("STDOUT:\n", r.stdout)
print("STDERR:\n", r.stderr)
assert r.returncode == 0, "Final C4 pipeline failed."
assert pred_file.exists(), "Expected predictions CSV not created."

## 8. Start Mage UI

In [ ]:
# Launch Mage in the background. Keep the Colab runtime running.
log_file = open("/content/mage_server.log", "w")
server = subprocess.Popen([MAGE, "start", "iris_project"], cwd="/content",
                          stdout=log_file, stderr=subprocess.STDOUT,
                          start_new_session=True)
time.sleep(20)
print("Mage server PID:", server.pid)
print(Path("/content/mage_server.log").read_text()[-4000:])

### Cloudflare tunnel and screenshot

The assignment requires the Mage UI through a Cloudflare tunnel. Start a Cloudflare quick tunnel targeting `http://localhost:6789` using the `cloudflared` executable in a separate Colab cell. Open the exact `trycloudflare.com` URL printed by the command. In Mage, open `iris_classifier` and capture the graph showing:

- `load_iris` → `validate_iris` → `prepare_data` → `train_model`
- `train_model` → `save_results`
- `train_model` → `evaluate_model`

Do not invent a tunnel URL. Save a screenshot of the actual graph for the evidence appendix.

## 9. Evidence checklist

In [12]:
checks = {
    "C1 load_iris": (PROJECT / "data_loaders/load_iris.py").exists(),
    "C1 prepare_data": (PROJECT / "transformers/prepare_data.py").exists(),
    "C1 train_model": (PROJECT / "transformers/train_model.py").exists(),
    "C1 save_results": (PROJECT / "data_exporters/save_results.py").exists(),
    "C3 evaluate_model": (PROJECT / "transformers/evaluate_model.py").exists(),
    "C4 validate_iris": (PROJECT / "transformers/validate_iris.py").exists(),
    "pipeline metadata": (pipeline_dir / "metadata.yaml").exists(),
    "predictions CSV": (PROJECT / "iris_predictions.csv").exists()
}
for k, v in checks.items():
    print(("OK      " if v else "MISSING "), k)
print("\nC2 actual experiment results:")
display(experiment_table)

OK       C1 load_iris
OK       C1 prepare_data
OK       C1 train_model
OK       C1 save_results
OK       C3 evaluate_model
OK       C4 validate_iris
OK       pipeline metadata
OK       predictions CSV

C2 actual experiment results:


NameError: name 'experiment_table' is not defined

In [13]:

from pathlib import Path
import pandas as pd

p = Path("/content/iris_project/iris_predictions.csv")

print("File exists:", p.exists())

if p.exists():
    df = pd.read_csv(p)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head())


File exists: True
Rows: 30
Columns: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target', 'split', 'predicted', 'test_accuracy', 'max_depth']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,split,predicted,test_accuracy,max_depth
0,5.0,3.4,1.5,0.2,0,test,0,0.966667,3
1,5.4,3.7,1.5,0.2,0,test,0,0.966667,3
2,5.8,4.0,1.2,0.2,0,test,0,0.966667,3
3,5.7,3.8,1.7,0.3,0,test,0,0.966667,3
4,5.4,3.4,1.7,0.2,0,test,0,0.966667,3


In [14]:

import subprocess

cmd = [
    "/content/mage_env/bin/mage",
    "run", "iris_project", "iris_classifier", "--test"
]

r = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)

print("Return code:", r.returncode)
print("\nLAST 5000 CHARACTERS OF ERROR:\n")
print(r.stderr[-5000:])


Return code: 1

LAST 5000 CHARACTERS OF ERROR:

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run



In [15]:

from pathlib import Path

save_file = Path("/content/iris_project/data_exporters/save_results.py")

save_code = '''
from pathlib import Path

if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter

if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test


@data_exporter
def export_data(df, *args, **kwargs):
    path = Path('/content/iris_project/iris_predictions.csv')
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} predictions to {path}")


@test
def test_export(*args, **kwargs):
    path = Path('/content/iris_project/iris_predictions.csv')
    assert path.exists(), "Predictions CSV was not created"
    assert path.stat().st_size > 0, "Predictions CSV is empty"
'''

save_file.write_text(save_code.strip() + "\n")
print(save_file.read_text())


from pathlib import Path

if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter

if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test


@data_exporter
def export_data(df, *args, **kwargs):
    path = Path('/content/iris_project/iris_predictions.csv')
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} predictions to {path}")


@test
def test_export(*args, **kwargs):
    path = Path('/content/iris_project/iris_predictions.csv')
    assert path.exists(), "Predictions CSV was not created"
    assert path.stat().st_size > 0, "Predictions CSV is empty"



In [16]:

import subprocess

cmd = [
    "/content/mage_env/bin/mage",
    "run", "iris_project", "iris_classifier", "--test"
]

result = subprocess.run(
    cmd, cwd="/content",
    capture_output=True, text=True
)

print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode == 0:
    print("C1 PIPELINE SUCCESSFUL")


Return code: 0
STDOUT:
 DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 | Test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows=true, columns=predicted):
    0  1   2
0  10  0   0
1   0  9   1
2   0  0  10
Metrics:
 accuracy  precision_macro  recall_macro  f1_macro
 0.966667         0.969697      0.966667  0.966583
Saved 30 predictions to /content/iris_project/iris_predictions.csv
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.

STDERR:
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run

C1 PIPELINE SUCCESSFUL


In [17]:

import subprocess
import json
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/iris_project")
MAGE = "/content/mage_env/bin/mage"

results = []

for depth in [1, 2, 3, 4, 5]:

    csv_file = PROJECT / "iris_predictions.csv"

    # Delete previous predictions before each run
    if csv_file.exists():
        csv_file.unlink()

    cmd = [
        MAGE,
        "run",
        "iris_project",
        "iris_classifier",
        "--test",
        "--runtime-vars",
        json.dumps({"max_depth": depth})
    ]

    print(f"\n{'='*50}")
    print(f"Running max_depth = {depth}")
    print("="*50)

    run = subprocess.run(
        cmd,
        cwd="/content",
        capture_output=True,
        text=True
    )

    print(run.stdout)
    if run.stderr:
        print("STDERR:", run.stderr[-1500:])

    file_created = csv_file.exists()
    accuracy = None

    if file_created:
        predictions = pd.read_csv(csv_file)
        if "test_accuracy" in predictions.columns:
            accuracy = float(predictions["test_accuracy"].iloc[0])

    results.append({
        "max_depth": depth,
        "test_accuracy": accuracy,
        "test_passed": run.returncode == 0,
        "csv_created": file_created
    })

results_df = pd.DataFrame(results)

print("\nFINAL C2 EXPERIMENT RESULTS")
display(results_df)



Running max_depth = 1
DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth=1 | Test accuracy=0.6667
FAIL: test_accuracy (block: train_model)
--------------------------------------------------------------
Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/executors/pipeline_executor.py", line 82, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/models/block/__init__.py", line 3350, in run_tests
    test_function(*outputs)
  File "<string>", line 30, in test_accuracy
AssertionEr

,max_depth,test_accuracy,test_passed,csv_created
0,1,NaN,False,False
1,2,0.933333,True,True
2,3,0.966667,True,True
3,4,0.933333,True,True
4,5,0.933333,True,True


In [18]:

from pathlib import Path

PROJECT = Path("/content/iris_project")

print("Evaluation block exists:",
      (PROJECT / "transformers/evaluate_model.py").exists())

print("Validation block exists:",
      (PROJECT / "transformers/validate_iris.py").exists())

print("\nFinal pipeline metadata:\n")
print((PROJECT / "pipelines/iris_classifier/metadata.yaml").read_text())


Evaluation block exists: True
Validation block exists: True

Final pipeline metadata:

blocks:
- name: load_iris
  type: data_loader
  uuid: load_iris
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  all_upstream_blocks_executed: true
  status: not_executed
- name: prepare_data
  type: transformer
  uuid: prepare_data
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: train_model
  type: transformer
  uuid: train_model
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: save_results
  type: data_exporter
  uuid: save_results
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
- name: evaluate_model
  type: transformer
  uuid: evaluate_model
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_b

In [19]:

from pathlib import Path
import yaml

PROJECT = Path("/content/iris_project")
metadata_path = PROJECT / "pipelines/iris_classifier/metadata.yaml"

metadata = yaml.safe_load(metadata_path.read_text())

# Connect load_iris -> validate_iris -> prepare_data
for block in metadata["blocks"]:
    if block["uuid"] == "load_iris":
        block["downstream_blocks"] = ["validate_iris"]

    elif block["uuid"] == "prepare_data":
        block["upstream_blocks"] = ["validate_iris"]

# Add the custom validation block if not already present
if not any(b["uuid"] == "validate_iris" for b in metadata["blocks"]):
    metadata["blocks"].append({
        "name": "validate_iris",
        "type": "transformer",
        "uuid": "validate_iris",
        "upstream_blocks": ["load_iris"],
        "downstream_blocks": ["prepare_data"],
        "all_upstream_blocks_executed": True,
        "status": "not_executed"
    })

metadata_path.write_text(
    yaml.safe_dump(metadata, sort_keys=False)
)

print(metadata_path.read_text())


blocks:
- name: load_iris
  type: data_loader
  uuid: load_iris
  upstream_blocks: []
  downstream_blocks:
  - validate_iris
  all_upstream_blocks_executed: true
  status: not_executed
- name: prepare_data
  type: transformer
  uuid: prepare_data
  upstream_blocks:
  - validate_iris
  downstream_blocks:
  - train_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: train_model
  type: transformer
  uuid: train_model
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
  all_upstream_blocks_executed: true
  status: not_executed
- name: save_results
  type: data_exporter
  uuid: save_results
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
- name: evaluate_model
  type: transformer
  uuid: evaluate_model
  upstream_blocks:
  - train_model
  downstream_blocks: []
  all_upstream_blocks_executed: true
  status: not_executed
- name: validate_iris
  type: transfor

In [20]:

import subprocess, json

cmd = [
    "/content/mage_env/bin/mage",
    "run", "iris_project", "iris_classifier",
    "--test",
    "--runtime-vars", json.dumps({"max_depth": 3})
]

result = subprocess.run(
    cmd, cwd="/content",
    capture_output=True, text=True
)

print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

assert result.returncode == 0, "Final pipeline run failed."


Return code: 0
STDOUT:
 DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Validation passed: no missing values and exactly 3 classes.
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth=3 | Test accuracy=0.9667
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows=true, columns=predicted):
    0  1   2
0  10  0   0
1   0  9   1
2   0  0  10
Saved 30 predictions to /content/iris_project/iris_predictions.csv
Metrics:
 accuracy  precision_macro  recall_macro  f1_macro
 0.966667         0.969697      0.966667  0.966583
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.

STDERR:
INFO:ma

In [21]:

import subprocess
import time
from pathlib import Path

MAGE = "/content/mage_env/bin/mage"

log = open("/content/mage_ui.log", "w")

server = subprocess.Popen(
    [MAGE, "start", "iris_project"],
    cwd="/content",
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(20)

print("Mage process ID:", server.pid)
print(Path("/content/mage_ui.log").read_text()[-3000:])


Mage process ID: 12922
lidation.
INFO:alembic.runtime.migration:Running upgrade 83a74eca6ea1 -> e7beb59b44f9, Add repo path to pipeline schedule
INFO:alembic.runtime.migration:Running upgrade e7beb59b44f9 -> dfe49d040487, Add condition_failed status to block run status
INFO:alembic.runtime.migration:Running upgrade dfe49d040487 -> 90d978a8aef8, Update unique constraint for secret
[IPKernelApp] CRITICAL | Bad config encountered during initialization: The 'kernel_class' trait of <ipykernel.kernelapp.IPKernelApp object at 0x78336f9efce0> instance must be a type, but 'google.colab._kernel.Kernel' could not be imported
INFO:alembic.runtime.migration:Running upgrade 90d978a8aef8 -> e3593cc2191e, Add key uuid to secrets
INFO:alembic.runtime.migration:Running upgrade e3593cc2191e -> 66928ca5b0d7, Add global_data_product_uuid to PipelineSchedule
INFO:alembic.runtime.migration:Running upgrade 66928ca5b0d7 -> 386bcfebd48d, Create Tag and TagAssociation tables
INFO:alembic.runtime.migration:Runnin

In [22]:

import subprocess

subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/content/cloudflared"
], check=True)

subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

print("Cloudflare tunnel client installed.")


Cloudflare tunnel client installed.


In [30]:
import time
from pathlib import Path

for i in range(6):
    time.sleep(5)
    output = Path("/content/cloudflared.log").read_text(errors="replace")
    print(f"\n--- Check {i+1} ---")
    print(output[-1500:])

    if "trycloudflare.com" in output:
        print("\nPUBLIC URL FOUND")
        break


--- Check 1 ---
-----------------------------------------------------------------+
2026-11-15T10:53:39Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=b72578af-8349-4053-ac4f-f5c999b1f9a5 status=pass target=region1.v2.argotunnel.com
2026-11-15T10:53:39Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=b72578af-8349-4053-ac4f-f5c999b1f9a5 status=pass target=region2.v2.argotunnel.com
2026-11-15T10:53:39Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=b72578af-8349-4053-ac4f-f5c999b1f9a5 status=pass target=region1.v2.argotunnel.com
2026-11-15T10:53:39Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=b72578af-8349-4053-ac4f-f5c999b1f9a5 status=pass target=region2.v2.argotunnel.com
2026-11-15T10:53:39Z INF precheck component="TCP Connectivity" details="HTTP/2 connection successful" run_id=b72578af-8349-4053-ac4f-f5c999b1f9a5 status=pass target=r

In [33]:
import re
from pathlib import Path

output = Path("/content/cloudflared.log").read_text(errors="replace")

urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', output)

if urls:
    print("Your Mage UI URL:")
    print(urls[-1])
else:
    print("Actual URL not generated yet.")
    print(output[-3000:])

Your Mage UI URL:
https://cancellation-allocation-repeated-pat.trycloudflare.com


In [37]:
import sys
sys.path.insert(0, "/content/iris_project")

import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Recreate the same deterministic Iris test split
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reproduce the successful max_depth=3 model
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

print("\nEVALUATION METRICS")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro Precision:", precision_score(y_test, y_pred, average="macro"))
print("Macro Recall:", recall_score(y_test, y_pred, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

CONFUSION MATRIX
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

EVALUATION METRICS
Accuracy: 0.9666666666666667
Macro Precision: 0.9696969696969697
Macro Recall: 0.9666666666666667
Macro F1: 0.9665831244778612
